# VitaCall — ML Engineering & Ops

Groepsrapport — Jupyter Notebook

---

In [ ]:
# !pip install pyspark scikit-learn pandas numpy mlflow fastapi uvicorn requests

## 1. Datapipeline (Week 6)

### 1.1 Data-ingestie
Ruwe IMDb-reviews downloaden en opslaan als Parquet.

In [ ]:
import os, sys
import pandas as pd
sys.path.insert(0, '..')
from pipeline.data import download_imdb, ingest_imdb

base_dir   = '../data'
bronze_dir = os.path.join(base_dir, 'bronze', 'imdb')
os.makedirs(bronze_dir, exist_ok=True)

imdb_dir = download_imdb(base_dir)
ingest_imdb(imdb_dir, bronze_dir)

df_bronze = pd.read_parquet(os.path.join(bronze_dir, 'imdb.parquet'))
print(f'Rijen: {len(df_bronze)}')
df_bronze.head(3)

### 1.2 Data cleaning
HTML-tags verwijderen, lege teksten en duplicaten eruit filteren.

In [ ]:
from pipeline.data import clean_reviews

silver_dir = os.path.join(base_dir, 'silver', 'imdb')
clean_reviews(os.path.join(bronze_dir, 'imdb.parquet'), silver_dir)

df_silver = pd.read_parquet(silver_dir)
print(f'Silver rijen: {len(df_silver)}')
df_silver[['text_clean', 'label', 'split']].head(3)

### 1.3 Feature engineering
Stratified train/val/test splits met PySpark.

In [ ]:
from pipeline.data import get_spark, create_features

gold_dir = os.path.join(base_dir, 'gold', 'imdb')
os.makedirs(gold_dir, exist_ok=True)

spark = get_spark()
spark.sparkContext.setLogLevel('WARN')
create_features(spark, silver_dir, gold_dir)
spark.stop()

df_gold = pd.read_parquet(gold_dir)
print(df_gold.groupby(['split', 'label']).size().reset_index(name='count').to_string())
df_gold.head(3)

---

## 2. Modelleren & Model Tracking (Week 12)

### 2.1 Model training
TF-IDF + Logistic Regression op de train-split.

In [ ]:
from pipeline.data import train

model_path = '../pipeline/sentiment_model.pkl'
df_tr  = df_gold[df_gold['split'] == 'train']
df_val = df_gold[df_gold['split'] == 'val']
df_te  = df_gold[df_gold['split'] == 'test']

metrics = train(
    df_tr['text_clean'].tolist(),  df_tr['label'].tolist(), model_path,
    val_texts=df_val['text_clean'].tolist(), val_labels=df_val['label'].tolist()
)
print('Validatie metrics:', metrics)

### 2.2 Model tracking met MLflow
Parameters, metrics en artefacten worden automatisch gelogd tijdens `train()`.

In [ ]:
import mlflow

runs = mlflow.search_runs(order_by=['start_time DESC'])
runs[['run_id', 'metrics.accuracy', 'metrics.f1', 'params.n_samples']].head(5)

### 2.3 Federatief leren
FedAvg: modelgewichten middelen over meerdere clients (privacy-preserving).

In [ ]:
from pipeline.data import federated_train

texts  = df_tr['text_clean'].tolist()
labels = df_tr['label'].tolist()
mid    = len(texts) // 2

clients = [(texts[:mid], labels[:mid]), (texts[mid:], labels[mid:])]
federated_train(clients, '../pipeline/sentiment_model_fed.pkl', rounds=3)
print('Federatief model opgeslagen.')

---

## 3. Deployment (Week 17)

### 3.1 API deployment (FastAPI)
Start de API met `uvicorn pipeline.api:app --reload` of via Docker.

In [ ]:
import requests

resp = requests.post('http://localhost:8000/analyze',
                     json={'text': 'ik heb pijn op de borst'})
print(resp.json())

### 3.2 Edge deployment (Electron)
Offline sentiment-analyse in de Electron-app zonder API-verbinding.

In [ ]:
NEG = ['pijn','benauwd','bloeding','bewusteloos','misselijk','koorts',
       'hartaanval','niet ademen','ernstig','help','gevallen','overdosis']
POS = ['goed','prima','rustig','stabiel','geen pijn','oké','normaal','kalm']

def edge_sentiment(text):
    t = text.lower()
    score = sum(w in t for w in POS) - sum(w in t for w in NEG)
    confidence = min(0.50 + abs(score) * 0.08, 0.90)
    return {'sentiment': 'positief' if score >= 0 else 'negatief',
            'confidence': confidence, 'source': 'edge'}

print(edge_sentiment('ik heb veel pijn'))
print(edge_sentiment('alles goed, stabiel'))

### 3.3 Monitoring & Drift detectie
De API houdt een sliding window van 100 voorspellingen bij.

In [ ]:
drift = requests.get('http://localhost:8000/drift').json()
print(f"Status:         {drift['status']}")
print(f"Positief rate:  {drift['positive_rate']}")
print(f"Drift score:    {drift['drift_score']}")
print(f"Samples:        {drift['samples']}")

---

## Bronvermelding

- IMDb dataset: Maas, A.L. et al. (2011). *Learning Word Vectors for Sentiment Analysis.*
- scikit-learn: Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python.*
- MLflow: https://mlflow.org/
- PySpark: https://spark.apache.org/docs/latest/api/python/
- FastAPI: https://fastapi.tiangolo.com/
- Vosk: https://alphacephei.com/vosk/